# Week 43 extension: calibrated abstention

**Hypothesis (stated before running).** The integrated QA model uses one global no-answer threshold, but the
validation languages differ strongly in how often questions are unanswerable (Arabic 12.5%, Korean 5.3%,
Telugu 24.2%) and the training split differs again (Telugu 3.3%). We expect the no-answer score to be shifted
differently per language, so **language-specific thresholds should beat a single global threshold**.

- Primary metric: overall answerability macro-F1 under cross-validation within the validation set.
- Expected direction: per-language > global, by at least 0.01 macro-F1 overall, with the gain concentrated in
  Telugu (higher unanswerable recall) and answerable recall falling by less than 0.03.
- **Falsified if** the overall gain is below 0.01, or Telugu does not improve, or per-language is not better in
  most repeats.
- Secondary question: does a language-specific intercept in Platt scaling reduce ECE / Brier score?

Protocol: 5-fold stratified (language x label) cross-validation inside validation, repeated 5 times with fixed
seeds. Each threshold is fitted on four folds and scored on the held-out fold; out-of-fold predictions are pooled
per repeat. The challenge set is not used.

In [5]:
from notebooks import nb_setup
import pandas as pd
from src.calibration import (
    cross_validate, summarise, paired_difference, evaluate_at_threshold, best_threshold,
)

df = pd.read_csv("../data/qa_val_predictions.csv", keep_default_na=False)
df["gold_answerable"] = df["gold_answerable"].astype(int)
print(len(df), "examples;", df.groupby("lang")["gold_answerable"].agg(["size", "mean"]).round(3).to_dict("index"))

1155 examples; {'ar': {'size': 415, 'mean': 0.875}, 'ko': {'size': 356, 'mean': 0.947}, 'te': {'size': 384, 'mean': 0.758}}


## Reference: the notebook-7 threshold, fitted and scored on all of validation (optimistic)

In [6]:
evaluate_at_threshold(df, -5.895390)

,group,n,macro_f1,unans_precision,unans_recall,unans_f1,ans_recall
0,Overall,1155,0.8478,0.7453,0.7317,0.7385,0.9586
1,ar,415,0.9381,0.8333,0.9615,0.8929,0.9725
2,ko,356,0.8064,0.5357,0.7895,0.6383,0.9614
3,te,384,0.7848,0.7534,0.5914,0.6627,0.9381


## Cross-validated comparison

In [7]:
results = cross_validate(df, n_splits=5, n_repeats=5, seed=42)
decision = results[results["method"].isin(["global", "per_language"])]
summarise(decision, ["macro_f1", "unans_precision", "unans_recall", "ans_recall", "squad_em", "squad_f1"])

macro_f1         unans_precision         unans_recall  \
                         mean     std            mean     std         mean   
method       group                                                           
global       Overall   0.8559  0.0014          0.8556  0.0060       0.6646   
             ar        0.9564  0.0022          0.8897  0.0070       0.9615   
             ko        0.8482  0.0000          0.6522  0.0000       0.7895   
             te        0.7692  0.0014          0.9129  0.0084       0.4731   
per_language Overall   0.8601  0.0027          0.7983  0.0180       0.7220   
             ar        0.9553  0.0029          0.8893  0.0068       0.9577   
             ko        0.8818  0.0157          0.7845  0.0111       0.7684   
             te        0.7749  0.0074          0.7331  0.0306       0.5806   

                             ans_recall         squad_em         squad_f1  \
                         std       mean     std     mean     std     mean   
method       group                                                          
global       Overall  0.0000     0.9814  0.0009   0.4926  0.0000   0.5876   
             ar       0.0000     0.9829  0.0012   0.5157  0.0000   0.6145   
             ko       0.0000     0.9763  0.0000   0.5337  0.0000   0.6421   
             te       0.0000     0.9856  0.0015   0.4297  0.0000   0.5080   
per_language Overall  0.0092     0.9697  0.0038   0.4989  0.0011   0.5918   
             ar       0.0086     0.9829  0.0012   0.5152  0.0011   0.6140   
             ko       0.0471     0.9881  0.0000   0.5326  0.0025   0.6415   
             te       0.0132     0.9320  0.0118   0.4500  0.0029   0.5216   

                              
                         std  
method       group            
global       Overall  0.0000  
             ar       0.0000  
             ko       0.0000  
             te       0.0000  
per_language Overall  0.0007  
             ar       0.0011  
             ko       0.0025  
             te       0.0025

In [8]:
for metric in ["macro_f1", "unans_recall", "ans_recall", "squad_f1"]:
    print(metric, "(per_language - global)")
    display(paired_difference(decision, "per_language", "global", metric))

macro_f1 (per_language - global)


,mean,std,repeats_better,repeats
group,,,,
Overall,0.0042,0.0033,4,5
ar,-0.0011,0.0043,1,5
ko,0.0336,0.0157,5,5
te,0.0057,0.0083,4,5


unans_recall (per_language - global)


,mean,std,repeats_better,repeats
group,,,,
Overall,0.0573,0.0092,5,5
ar,-0.0038,0.0086,0,5
ko,-0.0211,0.0471,0,5
te,0.1075,0.0132,5,5


ans_recall (per_language - global)


,mean,std,repeats_better,repeats
group,,,,
Overall,-0.0117,0.0043,0,5
ar,0.0000,0.0019,1,5
ko,0.0119,0.0000,5,5
te,-0.0536,0.0125,0,5


squad_f1 (per_language - global)


,mean,std,repeats_better,repeats
group,,,,
Overall,0.0042,0.0007,5,5
ar,-0.0005,0.0011,0,5
ko,-0.0006,0.0025,4,5
te,0.0136,0.0025,5,5


## Calibration quality (lower is better)

In [9]:
calibration = results[results["method"].str.startswith("platt")]
summarise(calibration, ["ece", "brier"])

ece           brier        
                              mean     std    mean     std
method             group                                  
platt_global       Overall  0.0245  0.0011  0.0580  0.0002
                   ar       0.0559  0.0023  0.0276  0.0002
                   ko       0.0420  0.0023  0.0301  0.0002
                   te       0.0695  0.0026  0.1167  0.0003
platt_per_language Overall  0.0243  0.0010  0.0556  0.0002
                   ar       0.0477  0.0012  0.0276  0.0002
                   ko       0.0169  0.0023  0.0240  0.0002
                   te       0.0613  0.0044  0.1150  0.0003

In [10]:
paired_difference(calibration, "platt_per_language", "platt_global", "ece")

,mean,std,repeats_better,repeats
group,,,,
Overall,-0.0002,0.0009,2,5
ar,-0.0081,0.0020,0,5
ko,-0.0251,0.0032,0,5
te,-0.0082,0.0035,0,5


## Interpretation

**Outcome: the hypothesis is not supported at the pre-stated size.**

| Criterion (stated before running) | Observed |
|---|---|
| Overall macro-F1 gain ≥ 0.01 | +0.0042 (std 0.0033 over 5 repeats); per-language better in 4/5 repeats |
| Gain concentrated in Telugu | Telugu +0.0057 (std 0.0083), within noise; the gain is in Korean (+0.034, 5/5 repeats); Arabic −0.001 |
| Answerable recall drops < 0.03 | Overall −0.012 (met); Telugu −0.054 (not met) |

The overall gain is below the 0.01 threshold, so we treat the hypothesis as falsified.

**What the thresholds did.** The fitted thresholds on all of validation are global −2.03, Arabic −1.97,
Korean +0.98, Telugu −5.41. The Telugu threshold is more conservative, as expected: Telugu unanswerable recall
rises by 0.108 (0.473 → 0.581). But unanswerable precision falls from 0.91 to 0.73 and answerable recall falls by
0.054, so macro-F1 barely moves (+0.006). Per-language thresholds trade errors against each other in Telugu
rather than removing them. SQuAD-v2 token F1 improves slightly (overall +0.004, Telugu +0.014).

**Korean.** The largest macro-F1 gain is in Korean, which we did not predict. Validation has only 19 unanswerable
Korean questions (about 15 per training fold), and the repeats reshuffle the same 356 examples, so "5/5 repeats"
is not independent evidence. We treat it as unreliable.

**Calibration.** A language-specific Platt intercept lowers ECE in each language (Arabic −0.008, Korean −0.025,
Telugu −0.008; never worse in any repeat) and lowers overall Brier score (0.0580 → 0.0556), but overall ECE is
unchanged (−0.0002). (In the `paired_difference` ECE table, `repeats_better` = 0 means per-language was never
worse, because lower ECE is better.) Better probabilities did not translate into clearly better abstention.

**Tuning objective.** The notebook-7 threshold (−5.90) was tuned for token F1. A global threshold tuned for
macro-F1 is −2.03 and reaches 0.856 cross-validated macro-F1, against 0.848 for −5.90 evaluated on all of
validation. The choice of objective matters about as much as per-language thresholds. This compares different
protocols, so it is indicative only.

**Limitations.** One trained QA model (no training-seed variation); the repeats vary only the fold split; small
Korean and Telugu unanswerable counts; no significance test, so we make no claim of statistical significance.
Even the best variant (0.860) is below the Week 39 GRU (0.878).

In [11]:
from src.calibration import PerLanguageThreshold, GlobalThreshold
y, s, l = df["gold_answerable"].values, df["no_answer_score"].values, df["lang"].values
print("global:", round(GlobalThreshold().fit(s, y, l).threshold, 3))
print("per language:", {k: round(v, 3) for k, v in PerLanguageThreshold().fit(s, y, l).thresholds.items()})

global: -2.032
per language: {'ar': -1.967, 'ko': 0.975, 'te': -5.413}
